### ingest_daily_support_tickets

In [1]:
 pip install --upgrade pip

In [2]:
pip install boto3 

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install sqlalchemy datetime dotenv

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install --upgrade pymysql

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import pandas as pd
import boto3
from io import StringIO
from sqlalchemy import create_engine
from datetime import datetime, timedelta

from dotenv import load_dotenv

load_dotenv()

True

In [6]:
# ---------- CONFIG ----------
db_config = {
    "host": "localhost",
    "port": "3306",
    "user": "root",  # change
    "password": "root", # change
    "database": "careplus_support_db"
}

S3_BUCKET = "careplus-data0" 
S3_PREFIX = "support-tickets/raw/"  
DATE_TRACKER_FILE = "date_tracker.txt"

import os

AWS_CONFIG = {
    "aws_access_key_id": os.getenv("AWS_ACCESS_KEY"),
    "aws_secret_access_key": os.getenv("SECRET_KEY"),
    "region_name": os.getenv("REGION")
}

In [7]:
pip install pymysql

Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install cryptography

Note: you may need to restart the kernel to use updated packages.


In [11]:
# ---------- UTILITY FUNCTIONS ----------
def get_engine(config):
    return create_engine(f"mysql+pymysql://{config['user']}:{config['password']}@{config['host']}:{config['port']}/{config['database']}")

def upload_to_s3(df, bucket, key):
    csv_buffer = StringIO()
    df.to_csv(csv_buffer, index=False)

    s3 = boto3.client('s3', **AWS_CONFIG)
    s3.put_object(Bucket=bucket, Key=key, Body=csv_buffer.getvalue())
    print(f"✅ Uploaded to s3://{bucket}/{key}")

def read_last_date(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            return f.read().strip()
    return "2025-06-30"  # Starting point before 1st July

def update_last_date(file_path, new_date):
    with open(file_path, 'w') as f:
        f.write(new_date)

def get_next_date(last_date_str):
    last_date = datetime.strptime(last_date_str, "%Y-%m-%d")
    next_date = last_date + timedelta(days=1)
    return next_date.strftime("%Y-%m-%d")

# ---------- MAIN INGESTION LOGIC ----------
def run_ingestion():
    engine = get_engine(db_config)
    last_date = read_last_date(DATE_TRACKER_FILE)
    next_date = get_next_date(last_date)

    # Query only that day’s data
    query = f"""
        SELECT * FROM support_tickets
        WHERE DATE(created_at) = '{next_date}';
    """
    df = pd.read_sql(query, engine)
    print(df.shape)
    print(df.head())

    if df.empty:
        print(f"⚠️ No data found for {next_date}. Skipping upload.")
        return

    # Upload to S3
    s3_key = f"{S3_PREFIX}support_tickets_{next_date}.csv"
    upload_to_s3(df, S3_BUCKET, s3_key)

    # Update date tracker
    update_last_date(DATE_TRACKER_FILE, next_date)
    print(f"📅 Updated tracker to {next_date}")

# Run
if __name__ == "__main__":
    run_ingestion()

(22, 10)
    ticket_id        created_at       resolved_at   agent priority  \
0  TCK0705000  2025-07-05 01:01  2025-07-05 20:44  Ananya   Medium   
1  TCK0705001  2025-07-05 01:31  2025-07-05 23:06   Kavya      Low   
2  TCK0705002  2025-07-05 01:53  2025-07-05 17:10   Sneha    Medum   
3  TCK0705002  2025-07-05 01:53  2025-07-05 17:10   Sneha    Medum   
4  TCK0705003  2025-07-05 03:45  2025-07-06 09:57   Sneha   Medium   

  num_interactions         IssUeCat channel    status agent_feedback  
0                6       Bug Report    Chat  Resolved                 
1                8      Login Issue   Phone  Resolved                 
2                2  Payment Failure    Chat  Resolved                 
3                2  Payment Failure    Chat  Resolved                 
4                6  Feature Request   Phone  Resolved                 
✅ Uploaded to s3://careplus-data0/support-tickets/raw/support_tickets_2025-07-05.csv
📅 Updated tracker to 2025-07-05
